# 003 Extract dark-matter candidates

Canonical candidate-extraction stage for the paper pipeline.


## Purpose, inputs, and outputs

**Purpose**
- Clean the canonical paper abstracts for model extraction.
- Identify dark-matter candidate mentions.
- Build the lean paper-level and candidate-level outputs used by the manuscript figures.

**Reads**
- `data/papers.parquet`
- `data/paper_arxiv_classes.parquet`

**Writes**
- `data/papers_with_dm_models.parquet`
- `data/dm_model_candidates_long.parquet`
- `code/stage-outputs/003-extract-dm-candidates/dm_model_mentions.parquet`
- `code/stage-outputs/003-extract-dm-candidates/dm_model_counts_by_year.parquet`

<br>

---

## Setup and workspace rooting

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
CODE_DIR = WORKSPACE_DIR / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
PAPER_DIR = paths["paper_dir"]
PAPER_OUTPUTS_DIR = paths["paper_outputs"]
OVERLEAF_DIR = paths["overleaf"]

print("workspace_root:", WORKSPACE_DIR)


In [ ]:
from pathlib import Path

import ast
import importlib
import pandas as pd
import re

import dm_term_normalization as dmtn
importlib.reload(dmtn)

from dm_term_normalization import (
    CANDIDATE_TAGS,
    COMPILED,
    DM_TAGS_INFO,
    PATTERNS,
    SPECIES_LABEL_MAP,
    candidate_species_labels,
    categorize_arxiv,
    dm_nearby_in_window,
    expand_dm_tags,
    extract_dm_tags_with_spans,
    filter_candidate_tags,
    normalize_segment,
    normalize_text,
    primary_arxiv_class,
)

OUTPUT_DIR = LOCAL_DIR / "003-extract-dm-candidates"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SHARED_PROCESSED_DIR = DATA_DIR
PAPERS_PATH = SHARED_PROCESSED_DIR / "papers.parquet"
PAPER_CLASSES_PATH = SHARED_PROCESSED_DIR / "paper_arxiv_classes.parquet"
ENRICHED_PARQUET = DATA_DIR / "papers_with_dm_models.parquet"
CANDIDATE_LONG_PARQUET = DATA_DIR / "dm_model_candidates_long.parquet"
MENTIONS_PARQUET = OUTPUT_DIR / "dm_model_mentions.parquet"
COUNTS_PARQUET = OUTPUT_DIR / "dm_model_counts_by_year.parquet"

EARLIEST_YEAR = 1950
END_YEAR = 2025
BAD_DOC = [
    "pressrelease", "mastersthesis", "misc", "techreport", "dataset", "proposal", "proceedings",
    "abstract", "newsletter", "obituary", "talk", "circular", "bookreview", "software",
    "editorial", "erratum", "phdthesis",
]
BIOLOGY_TERMS_STRICT = {
    "microbial", "bacterial", "genome", "genomes", "eukaryotic", "genomic",
    "microbes", "microorganisms", "bacteria", "metabolites", "phyla", "mags", "metagenomics"
}
BIOLOGY_PATTERN = re.compile(r"\b(?:%s)\b" % "|".join(map(re.escape, BIOLOGY_TERMS_STRICT)), flags=re.I)

if not PAPERS_PATH.exists():
    raise FileNotFoundError(f"Missing required input: {PAPERS_PATH}")
if not PAPER_CLASSES_PATH.exists():
    raise FileNotFoundError(f"Missing required input: {PAPER_CLASSES_PATH}")

papers = pd.read_parquet(PAPERS_PATH)
paper_classes = pd.read_parquet(PAPER_CLASSES_PATH)

class_lists = (
    paper_classes.sort_values(["bibcode", "class_pos"], kind="mergesort")
    .groupby("bibcode", as_index=False)
    .agg(arxiv_class=("arxiv_class", list))
)

df = papers.merge(class_lists, on="bibcode", how="left")
df["arxiv_class"] = df["arxiv_class"].apply(lambda value: value if isinstance(value, list) else [])
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["doctype"] = df["doctype"].astype("string")
df["abstract"] = df["abstract"].astype("string")

year_mask = df["year"].notna() & df["year"].between(EARLIEST_YEAR, END_YEAR, inclusive="both")
abstract_mask = df["abstract"].notna()
doc_mask = ~df["doctype"].isin(BAD_DOC)
biology_mask = ~df["abstract"].str.contains(BIOLOGY_PATTERN, na=False)

eligible_mask = year_mask & abstract_mask & doc_mask & biology_mask

df["eligible_for_model_extraction"] = eligible_mask
df["abstract_clean"] = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[eligible_mask, "abstract_clean"] = (
    df.loc[eligible_mask, "abstract"]
      .astype("string")
      .str.lower()
      .map(lambda s: normalize_segment(s, mask_math=True))
      .astype("string")
)

print(f"Loaded papers: {len(df):,}")
print(f"Rows with abstracts: {abstract_mask.sum():,}")
print(f"Rows eligible for extraction: {eligible_mask.sum():,}")
print("dm_term_normalization module:", dmtn.__file__)


## Stage 1. Candidate-pattern extraction rules


In [ ]:
# Apply shared candidate ontology to the cleaned abstracts

if 'df' in globals() and isinstance(df, pd.DataFrame) and 'abstract_clean' in df.columns:
    df[['dm_models', 'dm_spans']] = df['abstract_clean'].apply(
        lambda s: pd.Series(extract_dm_tags_with_spans(s))
    )
    print(df[['abstract_clean', 'dm_models', 'dm_spans']].tail(20))


## Stage 2. Candidate labels and paper-level fields


In [ ]:
# Shared candidate labels and paper-level derived columns

if 'df' in globals() and isinstance(df, pd.DataFrame) and 'dm_models' in df.columns:
    df['primary_arxiv_class'] = df['arxiv_class'].apply(primary_arxiv_class)
    df['arxiv_category'] = df['primary_arxiv_class'].apply(categorize_arxiv)
    df['dm_models_full'] = df['dm_models'].apply(expand_dm_tags)
    df['dm_models_candidates'] = df['dm_models'].apply(filter_candidate_tags)
    df['dm_models_candidates_full'] = df['dm_models_candidates'].apply(expand_dm_tags)
    df['dm_candidate_species'] = df['dm_models_candidates'].apply(candidate_species_labels)
    df['dm_models_count'] = df['dm_models'].apply(lambda L: len(L) if isinstance(L, list) else 0)
    df['dm_models_candidates_count'] = df['dm_models_candidates'].apply(len)


## Stage 3. Mention-level audit table


In [ ]:
import ast
import pandas as pd

def ensure_list(obj):
    if isinstance(obj, list):
        return obj
    if pd.isna(obj):
        return []
    if isinstance(obj, str):
        try:
            v = ast.literal_eval(obj)
            return v if isinstance(v, list) else [obj]
        except Exception:
            return [obj]
    if isinstance(obj, tuple):
        return list(obj)
    return [obj]

df["dm_models_candidates_full"] = df["dm_models_candidates_full"].apply(ensure_list)
df["dm_spans"] = df["dm_spans"].apply(ensure_list)

df["dm_spans"] = df["dm_spans"].apply(lambda L: [tuple(x) for x in L])

def pair_tags_spans(row):
    tags, spans = row["dm_models_candidates_full"], row["dm_spans"]
    n = min(len(tags), len(spans))
    return list(zip(tags[:n], spans[:n]))

df["dm_pairs"] = df.apply(pair_tags_spans, axis=1)

tidy = df.explode("dm_pairs", ignore_index=True)
tidy = tidy.dropna(subset=["dm_pairs"])
tidy[["tag", "span"]] = tidy["dm_pairs"].apply(pd.Series)
tidy = tidy.drop(columns=["dm_pairs"])

def make_snippet(text, span, radius=30):
    if not isinstance(text, str) or not isinstance(span, (list, tuple)):
        return ""
    a, b = span
    a, b = max(0, a - radius), min(len(text), b + radius)
    return text[a:b]

tidy["snippet"] = tidy.apply(
    lambda r: make_snippet(r["abstract_clean"] if isinstance(r.get("abstract_clean"), str) else "", r["span"]),
    axis=1,
)

print(tidy[["tag", "span", "snippet"]].head(20))


## Stage 4. Write the canonical and audit outputs


In [ ]:
df_unique = tidy.drop_duplicates(subset=['bibcode', 'tag', 'year'])
df_unique = df_unique[df_unique['tag'] != 'dark matter']

paper_level_cols = [
    'bibcode', 'year', 'arxiv_class', 'primary_arxiv_class', 'arxiv_category',
    'eligible_for_model_extraction', 'abstract_clean',
    'dm_models', 'dm_models_candidates', 'dm_candidate_species',
    'dm_models_count', 'dm_models_candidates_count'
]
papers_with_models = df[paper_level_cols].copy()
papers_with_models.to_parquet(ENRICHED_PARQUET, index=False)

mention_cols = ['bibcode', 'year', 'arxiv_category', 'tag', 'span', 'snippet']
dm_mentions = df_unique[mention_cols].copy()
dm_mentions.to_parquet(MENTIONS_PARQUET, index=False)

candidate_long = (
    papers_with_models[['bibcode', 'year', 'arxiv_category', 'dm_models_candidates', 'dm_candidate_species']]
    .assign(candidate_pairs=lambda d: d.apply(
        lambda row: list(zip(row['dm_models_candidates'], row['dm_candidate_species']))
        if isinstance(row['dm_models_candidates'], list) and isinstance(row['dm_candidate_species'], list)
        else [],
        axis=1
    ))
    .drop(columns=['dm_models_candidates', 'dm_candidate_species'])
    .explode('candidate_pairs', ignore_index=True)
)
candidate_long = candidate_long.dropna(subset=['candidate_pairs']).copy()
candidate_long[['tag', 'SpeciesLabel']] = candidate_long['candidate_pairs'].apply(pd.Series)
candidate_long = candidate_long.drop(columns=['candidate_pairs']).drop_duplicates(
    subset=['bibcode', 'year', 'arxiv_category', 'tag', 'SpeciesLabel']
).reset_index(drop=True)
candidate_long.to_parquet(CANDIDATE_LONG_PARQUET, index=False)

paper_count_per_year = (
    df_unique.groupby(['tag', 'year'])
    .size()
    .reset_index(name='paper_count')
)
paper_count_per_year.to_parquet(COUNTS_PARQUET, index=False)

print(f"Saved lean paper-level parquet -> {ENRICHED_PARQUET}")
print(f"Saved mention-level parquet -> {MENTIONS_PARQUET}")
print(f"Saved candidate-long parquet -> {CANDIDATE_LONG_PARQUET}")
print(f"Saved yearly tag counts -> {COUNTS_PARQUET}")
print(f"Candidate rows: {len(candidate_long):,}")
print(f"Mention rows: {len(dm_mentions):,}")
print(f"Unique candidate labels: {candidate_long['SpeciesLabel'].nunique():,}")
display(candidate_long.head(10))
